# 10 — Model selection


## Imports


In [ ]:
from pathlib import Path
import hashlib
import json
from time import perf_counter
import numpy as np
import pandas as pd
import torch
import joblib
from torch import nn
import xgboost as xgb


## Configuration


In [ ]:
from pathlib import Path
# ใช้ project root เดียวกัน ไม่ต้อง import โค้ด pipeline จาก program
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'dataset').is_dir() and (p / 'model').is_dir() and (p / 'program').is_dir())
RAW = ROOT / 'dataset/raw/mit-bih-arrhythmia-database-1.0.0'
DATA = ROOT / 'dataset/processed'
ARTIFACTS = ROOT / 'model/artifacts'
REPORTS = ROOT / 'model/reports'
CLASSES = ['N', 'S', 'V', 'F', 'Q']
SYMBOLS = ['NLRej', 'AaJS', 'VE', 'F', '/fQ']
LABEL_MAP = {s: i for i, group in enumerate(SYMBOLS) for s in group}
SEED = 42
PREPROCESSING = {'version': 1, 'fs': 360, 'left': 90, 'right': 162,
                 'band': [0.5, 40], 'order': 4, 'leads': ['MLII', 'V5']}

## Artifact helpers


In [ ]:
def read_json(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'ไม่พบ {path}; กรุณารัน notebook ขั้นก่อนหน้าก่อน')
    return json.loads(path.read_text(encoding='utf-8'))

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def new_stage(marker):
    # ห้ามเขียนทับผลเดิมเงียบ ๆ เพราะอาจทำให้โมเดลอ้างอิงคนละ split
    if Path(marker).exists():
        raise FileExistsError(f'มีผลขั้นนี้แล้ว: {marker}; โหลดผลเดิมแทนการรันซ้ำ')
    if (ARTIFACTS / 'selection.json').exists():
        raise RuntimeError('ล็อกโมเดลแล้ว ไม่อนุญาตให้เตรียมข้อมูลหรือ tuning ใหม่ใน experiment เดิม')

## Load saved models


In [ ]:
class ECGNetwork(nn.Module):
    def __init__(self, kind, channels, kernel=7, hidden=64, dropout=.3):
        super().__init__()
        layers, previous = [], 1
        for width in channels:
            layers += [nn.Conv1d(previous, width, kernel, padding=kernel // 2),
                       nn.BatchNorm1d(width), nn.ReLU(), nn.MaxPool1d(2)]
            previous = width
        self.conv = nn.Sequential(*layers)
        self.kind = kind
        if kind == 'cnn_bilstm':
            self.lstm = nn.LSTM(previous, hidden, batch_first=True, bidirectional=True)
            previous = hidden * 2
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(previous, 5))

    def forward(self, x):
        x = self.conv(x)
        if self.kind == 'cnn_bilstm':
            _, (hidden, _) = self.lstm(x.transpose(1, 2))
            x = torch.cat([hidden[-2], hidden[-1]], dim=1)
        else:
            x = x.mean(dim=2)
        return self.head(x)

def network(kind, parameters):
    return ECGNetwork(kind, parameters['channels'], parameters.get('kernel', 7),
                      parameters.get('hidden', 64), parameters.get('dropout', .3))

def load_model(folder, info, device='cpu'):
    kind = info['kind']
    if kind in ['rf', 'dummy']:
        return joblib.load(folder / 'model.joblib')
    if kind == 'xgb':
        model = xgb.Booster()
        model.load_model(folder / 'model.ubj')
        return model
    model = network(kind, info['parameters'])
    model.load_state_dict(torch.load(folder / 'model.pt', map_location='cpu', weights_only=True))
    return model.to(device).eval()

def probabilities(model, kind, values, device='cpu', batch_size=128):
    if kind in ['rf', 'dummy']:
        output = np.zeros((len(values), 5), dtype=np.float32)
        output[:, np.asarray(model.classes_, dtype=int)] = model.predict_proba(values)
        return output
    if kind == 'xgb':
        return model.predict(xgb.DMatrix(values))
    outputs = []
    model.eval()
    with torch.inference_mode():
        for start in range(0, len(values), batch_size):
            # waveform: (beats, 252) -> Conv1d: (beats, 1, 252)
            batch = torch.as_tensor(values[start:start + batch_size, None, :], device=device)
            outputs.append(model(batch).softmax(dim=1).cpu().numpy())
    return np.concatenate(outputs)

def synchronize(device):
    if str(device).startswith('cuda'):
        torch.cuda.synchronize()

## Load partition helper


In [ ]:
def load_partition(split, *, allow_test=False):
    if split not in ['train', 'validation', 'test']:
        raise ValueError('Unknown partition')
    if split == 'test' and not allow_test:
        raise PermissionError('Test เปิดได้เฉพาะ final evaluation หลังล็อกโมเดล')
    manifest = read_json(DATA / 'prepared.json')
    folder = DATA / split
    for name in ['metadata.parquet', 'waveforms.npy', 'features.npy']:
        if sha256(folder / name) != manifest['files'][f'{split}/{name}']:
            raise ValueError(f'artifact เปลี่ยน: {split}/{name}')
    meta = pd.read_parquet(folder / 'metadata.parquet')
    waves, x = np.load(folder / 'waveforms.npy'), np.load(folder / 'features.npy')
    if len(meta) != len(waves) or len(meta) != len(x) or meta.beat_id.duplicated().any():
        raise ValueError('beat alignment ไม่ถูกต้อง')
    if not meta.split.eq(split).all() or set(meta.class_id) != set(range(5)):
        raise ValueError('partition หรือ class coverage ไม่ถูกต้อง')
    if not np.array_equal(meta.waveform_index.to_numpy(), np.arange(len(meta))):
        raise ValueError('waveform index ไม่ตรงกับ metadata')
    if waves.shape != (len(meta), 252) or not np.isfinite(waves).all() or not np.isfinite(x).all():
        raise ValueError('ขนาด waveform หรือค่าข้อมูลหลัง preprocessing ไม่ถูกต้อง')
    return {'meta': meta, 'wave': waves, 'x': x, 'y': meta.class_id.to_numpy()}

## Measure inference latency


In [ ]:
def measure_latency(model, kind, values, device, repeats=20):
    result = {}
    for size in [1, 128]:
        batch = values[:size]
        for _ in range(3):
            probabilities(model, kind, batch, device, batch_size=size)
        timings = []
        for _ in range(repeats):
            synchronize(device)
            start = perf_counter()
            probabilities(model, kind, batch, device, batch_size=size)
            synchronize(device)
            timings.append(perf_counter() - start)
        result[f'batch_{size}_seconds'] = float(np.median(timings))
        result[f'batch_{size}_actual_count'] = len(batch)
    return result

## Verify artifacts and load Validation


In [ ]:
new_stage(ARTIFACTS / 'selection.json')
manifest = read_json(DATA / 'prepared.json')
# ตรวจ state และ imputer ด้วย เพื่อให้ model bundle ใช้ preprocessing ที่ฝึกจริง
for name in ['preprocessing.json', 'imputer.joblib', 'split.json']:
    if sha256(DATA / name) != manifest['files'][name]:
        raise ValueError(f'Preprocessing เปลี่ยน: {name}')
validation = load_partition('validation')
names = [f'{kind}_{weight}' for kind in ['rf', 'xgb', 'cnn', 'cnn_bilstm']
         for weight in ['unweighted', 'weighted']]
records = []

## Compare saved candidates


In [ ]:
for name in names + ['dummy_unweighted']:
    chosen = read_json(ARTIFACTS / name / 'selected.json')
    path = ARTIFACTS / name / chosen['candidate']
    info = read_json(path / 'info.json')
    if info['prepared_hash'] != sha256(DATA / 'prepared.json'):
        raise ValueError(f'{name} ใช้ข้อมูลคนละ version')
    device = info['device']
    if device == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('ต้องวัด latency บน GPU เดียวกับ training')
    model = load_model(path, info, device)
    values = validation['wave'] if info['kind'] in ['cnn', 'cnn_bilstm'] else validation['x']
    latency = measure_latency(model, info['kind'], values, device)
    hashes = {p.name: sha256(p) for p in path.iterdir() if p.is_file()}
    records.append({**chosen, 'path': path.relative_to(ARTIFACTS).as_posix(),
                    'kind': info['kind'], 'weighted': info['weighted'], 'device': device,
                    'train_seconds': info['train_seconds'], 'latency': latency, 'files': hashes})
    del model

## Select winner and lock artifacts


In [ ]:
scores = records[:8]
best = max(r['validation_macro_f1'] for r in scores)
eligible = [r for r in scores if best - r['validation_macro_f1'] <= .005]
winner = min(eligible, key=lambda r: (r['latency']['batch_1_seconds'], r['configuration']))
lock = {'winner': winner['configuration'], 'selection_metric': 'validation_macro_f1',
        'tie_tolerance': .005, 'prepared_hash': sha256(DATA / 'prepared.json'),
        'preprocessing_hash': sha256(DATA / 'preprocessing.json'),
        'imputer_hash': sha256(DATA / 'imputer.joblib'), 'candidates': records,
        'seed': SEED, 'test_used_for_selection': False}
write_json(ARTIFACTS / 'selection.json', lock)
pd.DataFrame([{k: r[k] for k in ['configuration', 'candidate', 'validation_macro_f1']} for r in records])